**Note:** file paths in this notebook have been replaced with `/path/to/...` placeholders. Edit them to match your own system before running.

In [ ]:
# First, import some stuff
import numpy as np
import h5py
import nibabel as nib
from mvpa2.suite import Hyperalignment, Dataset
from scipy import stats
import pandas as pd
import os

In [ ]:
# Load some helpful index files
pheno_table = pd.read_csv('/path/to/data/restsync/hcp_7t_pheno.csv')
IDs         = pheno_table['Subject']

# Remove subject who are missing movie (or rest) data
all_excludes    = [66,124,126,130,135,183];
all_excludes[:] = [x - 1 for x in all_excludes]

all_excludes_rest    = [8,66,124,126,130,135,179,183];
all_excludes_rest[:] = [x - 1 for x in all_excludes_rest]

IDs_use_movie = IDs.tolist()
for index in sorted(all_excludes, reverse=True):
    del IDs_use_movie[index]
    
IDs_use_rest = IDs.tolist()
for index in sorted(all_excludes_rest, reverse=True):
    del IDs_use_rest[index]
    
movie_indices = list(range(len(IDs_use_movie)))
rest_indices  = list(range(len(IDs_use_rest)))


In [ ]:
import pdb
def load_and_process_data(filepath):
    with h5py.File(filepath, 'r') as f:
        arrays = {k: np.array(v) for k, v in f.items()}
    return [arr for arr in np.split(arrays['cha_mat'], arrays['cha_mat'].shape[0])]

def load_subject_data(indices, scan_id, base_path):
    all_data = []
    for subj in indices:
        all_data.append(stats.zscore(np.load(base_path.format(scan_id, subj+1))))
    return all_data

def load_parc_ids(parc_res_curr, medial_mask, parc_ids_path_format):
    current_parc   = parc_ids_path_format.format(parc_res_curr)
    parc_ids       = nib.load(current_parc)
    parc_ids_array = np.reshape(np.asarray(parc_ids.get_data()).T.flatten(), (64984, 1))
    return parc_ids_array[medial_mask == 1].flatten()

def perform_connectivity_hyperalignment(scan_ids, parc_res_currs, rest_indices, IDs_use_rest, gen_transforms, paths):
    medial_mask = np.genfromtxt(paths['medial_mask'], delimiter=',').astype(int)
    for scan_id in scan_ids:
        other_scan_id = 3 - scan_id

        for parc_res_curr in parc_res_currs:
            parc_ids_array = load_parc_ids(parc_res_curr, medial_mask, paths['parc_ids_format'])
            filepath       = paths['connectivity_data'].format(parc_res_curr, scan_id)
            parc_ids_array = load_parc_ids(parc_res_curr, medial_mask, paths['parc_ids_format'])
            max_parc       = int(max(parc_ids_array))

            if gen_transforms:
                arrays = load_and_process_data(filepath)
                generate_connectivity_hyperalignment_transformations(arrays, parc_ids_array, max_parc, scan_id, paths['connectivity_output'])

            all_data = load_subject_data(rest_indices, other_scan_id, paths['subject_data'])
            
            hyperalignment_transformed = apply_connectivity_hyperalignment(all_data, parc_ids_array, max_parc, scan_id, IDs_use_rest, paths['connectivity_output'])
            
            save_transformed_data(hyperalignment_transformed, parc_res_curr, other_scan_id, IDs_use_rest, paths['transformed_data'],parc_ids_array)

def generate_connectivity_hyperalignment_transformations(arrays, parc_ids_array, max_parc, scan_id, output_path):
    print('Getting hyperalignment transformations')
    for roi in range(max_parc):
        ha = Hyperalignment()
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp = [(np.squeeze(np.squeeze(x)[:,np.where(parc_ids_array == (roi+1))])) for x in arrays]
            all_data_temp = [Dataset(stats.zscore(np.delete(x, roi, axis=0))) for x in all_data_temp]
            hyperalign    = ha(all_data_temp)
            np.save(output_path.format(roi, scan_id), hyperalign, allow_pickle=True)

def apply_connectivity_hyperalignment(all_data, parc_ids_array, max_parc, scan_id, IDs_use_rest, output_path):
    hyperalignment_transformed = []
    for roi in range(max_parc):
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp     = [Dataset(x[:, parc_ids_array == (roi+1)]) for x in all_data]
            trans_matrix_path = output_path.format(roi, scan_id)
            hyperalign        = np.load(trans_matrix_path, allow_pickle=True)
            dss_aligned       = [h.forward(sd) for h, sd in zip(hyperalign, all_data_temp)]
            hyperalignment_transformed.append([d.samples for d in dss_aligned])
        else:
            hyperalignment_transformed.append([[]] * len(IDs_use_rest))
    return hyperalignment_transformed

def save_transformed_data(hyperalignment_transformed, parc_res_curr, other_scan_id, IDs_use_rest, output_path,parc_ids_array):
    for subj_index, subj in enumerate(IDs_use_rest):
        output = np.zeros((59412, hyperalignment_transformed[0][subj_index].shape[0]))
        for roi, transformed_data in enumerate(hyperalignment_transformed):
            if np.sum(parc_ids_array == roi+1) > 0:
                output[parc_ids_array == (roi+1), :] = transformed_data[subj_index].T
        save_path = output_path.format(parc_res_curr, other_scan_id, subj_index+1)
        np.save(save_path, output, allow_pickle=True)

def perform_piecewise_hyperalignment(scan_id, parc_res,movie_indices, IDs_use_movie, paths,gen_transforms):
    last_subj     = len(IDs_use_movie)
    medial_mask   = np.genfromtxt(paths['medial_mask'], delimiter=',').astype(int)
    other_scan_id = 2 if scan_id == 1 else 1

    for parc_res_curr in parc_res:
        parc_ids_array = load_parc_ids(parc_res_curr, medial_mask, paths['parc_ids_format'])
        max_parc       = int(max(parc_ids_array))

        if gen_transforms == 1:
            all_data = load_subject_data(movie_indices, scan_id, paths['subject_data'])
            generate_piecewise_hyperalignment_transformations(all_data, parc_ids_array, max_parc, scan_id, paths['piecewise_output'])
            all_data = []
            
        all_data_new               = load_subject_data(movie_indices, other_scan_id, paths['subject_data'])
        hyperalignment_transformed = apply_piecewise_hyperalignment(all_data_new, parc_ids_array, max_parc, scan_id, other_scan_id, last_subj, paths['piecewise_intermediate_output'])
        save_transformed_data(hyperalignment_transformed, parc_res_curr, other_scan_id, IDs_use_movie, paths['transformed_data'],parc_ids_array)

def generate_piecewise_hyperalignment_transformations(all_data, parc_ids_array, max_parc, scan_id, output_path):
    print('Generating piecewise hyperalignment transformations')
    for roi in range(max_parc):
        ha = Hyperalignment()
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp = [Dataset(x[:,np.where(parc_ids_array == (roi+1))[0]]) for x in all_data]
            hyperalign    = ha(all_data_temp) 
            np.save(output_path.format(roi, scan_id), hyperalign, allow_pickle=True)

def apply_piecewise_hyperalignment(all_data, parc_ids_array, max_parc, scan_id, other_scan_id, last_subj, output_path):
    print('Applying piecewise hyperalignment')
    hyperalignment_transformed = []
    for roi in range(max_parc):
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp = [Dataset(x[:, np.where(parc_ids_array == (roi+1))[0]]) for x in all_data]
            hyperalign    = np.load(output_path.format(roi, scan_id), allow_pickle=True)
            dss_aligned   = [h.forward(sd) for h, sd in zip(hyperalign, all_data_temp)]
            hyperalignment_transformed.append([d.samples for d in dss_aligned])
        else:
            hyperalignment_transformed.append([[]] * len(all_data))
            
    return hyperalignment_transformed
def transformation_matrices_exist(max_parc, scan_id, process_type, output_path):
    for roi in range(max_parc):
        if os.path.exists(output_path.format(roi, scan_id)):
            return True
    return False

connectivity_paths = {
    'medial_mask': '/path/to/data/isc_heritability/medial_mask.csv',
    'connectivity_data': '/path/to/data/isc_heritability/data/anatomical/inputs/gray/cha_mat_parc_{}_day_{}.mat',
    'subject_data': '/path/to/data/isc_heritability/data/anatomical/inputs/gray/anatomical_movie_scan_{}_subj_{}.npy',
    'parc_ids_format': '/path/to/downloads/ThomasYeoLab CBIG master stable_projects-brain_parcellation_Schaefer2018_LocalGlobal_Parcellations_HCP_fslr32k_cifti/Schaefer2018_{}Parcels_17Networks_order.dscalar.nii',
    'connectivity_output': '/path/to/data/isc_heritability/connectivity_outputs2/connectivity_transformationmatrix_{}_{}.npy',
    'transformed_data': '/path/to/data/isc_heritability/procrustes_transformed2/connectivity_transformed_parc_{}_scan_{}_subj_{}.npy'
}

piecewise_paths = {
    'medial_mask': '/path/to/data/isc_heritability/medial_mask.csv',
    'subject_data': '/path/to/data/isc_heritability/data/anatomical/inputs/gray/anatomical_movie_scan_{}_subj_{}.npy',
    'parc_ids_format': '/path/to/downloads/ThomasYeoLab CBIG master stable_projects-brain_parcellation_Schaefer2018_LocalGlobal_Parcellations_HCP_fslr32k_cifti/Schaefer2018_{}Parcels_17Networks_order.dscalar.nii',
    'piecewise_output': '/path/to/data/isc_heritability/piecewise_outputs2/piecewise_transformationmatrix_{}_{}.npy',
    'transformed_data': '/path/to/data/isc_heritability/procrustes_transformed2/piecewise_transformed_parc_{}_scan_{}_subj_{}.npy'
}

In [ ]:
import numpy as np
import nibabel as nib
import h5py
from scipy import stats
from mvpa2.suite import Hyperalignment, Dataset
import os

def load_and_process_data(filepath):
    with h5py.File(filepath, 'r') as f:
        arrays = {k: np.array(v, dtype='float32') for k, v in f.items()}
    return [arr.astype('float32') for arr in np.split(arrays['cha_mat'], arrays['cha_mat'].shape[0], axis=0)]

def load_subject_data(indices, scan_id, base_path):
    all_data = []
    for subj in indices:
        data = np.load(base_path.format(scan_id, subj+1)).astype('float32')
        data = stats.zscore(data, axis=None)  # Apply z-score across the whole array if needed
        all_data.append(data.astype('float32'))  # Ensure float32
    return all_data

def load_parc_ids(parc_res_curr, medial_mask, parc_ids_path_format):
    current_parc = parc_ids_path_format.format(parc_res_curr)
    parc_ids     = nib.load(current_parc)
    
    parc_ids_array = np.array(parc_ids.dataobj).astype('float32')
    parc_ids_array = np.reshape(parc_ids_array.flatten(), (64984, 1))
    return parc_ids_array[medial_mask == 1].flatten()


def generate_connectivity_hyperalignment_transformations(arrays, parc_ids_array, max_parc, scan_id, output_path):
    print('Getting hyperalignment transformations')
    for roi in range(max_parc):
        ha = Hyperalignment()
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp = [np.squeeze(x[:, np.where(parc_ids_array == (roi+1))]).astype('float32') for x in arrays]
            all_data_temp = [Dataset(stats.zscore(x, axis=1).astype('float32')) for x in all_data_temp]  # zscore and ensure float32
            hyperalign    = ha(all_data_temp)
            np.save(output_path.format(roi, scan_id), hyperalign, allow_pickle=True)

def apply_connectivity_hyperalignment(all_data, parc_ids_array, max_parc, scan_id, IDs_use_rest, output_path):
    hyperalignment_transformed = []
    for roi in range(max_parc):
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp     = [x[:, np.where(parc_ids_array == (roi+1))[0]].astype('float32') for x in all_data]
            all_data_temp     = [Dataset(x) for x in all_data_temp]
            trans_matrix_path = output_path.format(roi, scan_id)
            hyperalign        = np.load(trans_matrix_path, allow_pickle=True)
            dss_aligned       = [h.forward(sd) for h, sd in zip(hyperalign, all_data_temp)]
            hyperalignment_transformed.append([d.samples.astype('float32') for d in dss_aligned])
        else:
            hyperalignment_transformed.append([np.array([]).astype('float32')] * len(IDs_use_rest))
    return hyperalignment_transformed

def save_transformed_data(hyperalignment_transformed, parc_res_curr, other_scan_id, IDs_use_rest, output_path, parc_ids_array):
    for subj_index, subj in enumerate(IDs_use_rest):
        output = np.zeros((59412, hyperalignment_transformed[0][subj_index].shape[1]), dtype='float32')
        for roi, transformed_data in enumerate(hyperalignment_transformed):
            if np.sum(parc_ids_array == roi+1) > 0:
                output[parc_ids_array == (roi+1), :] = transformed_data[subj_index].T.astype('float32')
        save_path = output_path.format(parc_res_curr, other_scan_id, subj_index+1)
        np.save(save_path, output, allow_pickle=True)

# Additional functions for piecewise_hyperalignment and checks for existing transformations remain unchanged but should ensure that any data manipulation or loading also enforces float32.
def generate_piecewise_hyperalignment_transformations(all_data, parc_ids_array, max_parc, scan_id, output_path):
    print('Generating piecewise hyperalignment transformations')
    for roi in range(max_parc):
        ha = Hyperalignment()
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp = [Dataset(x[:,np.where(parc_ids_array == (roi+1))[0]].astype('float32')) for x in all_data]
            all_data      = []
            hyperalign    = ha(all_data_temp)
            np.save(output_path.format(roi, scan_id), hyperalign, allow_pickle=True)

def apply_piecewise_hyperalignment(all_data, parc_ids_array, max_parc, scan_id, other_scan_id, last_subj, output_path):
    print('Applying piecewise hyperalignment')
    hyperalignment_transformed = []
    for roi in range(max_parc):
        if np.sum(parc_ids_array == roi+1) > 0:
            all_data_temp     = [Dataset(x[:, np.where(parc_ids_array == (roi+1))[0]].astype('float32')) for x in all_data]
            trans_matrix_path = output_path.format(roi, scan_id)
            hyperalign        = np.load(trans_matrix_path, allow_pickle=True)
            dss_aligned       = [h.forward(sd) for h, sd in zip(hyperalign, all_data_temp)]
            hyperalignment_transformed.append([d.samples.astype('float32') for d in dss_aligned])
        else:
            hyperalignment_transformed.append([np.array([]).astype('float32')] * len(all_data))
    return hyperalignment_transformed

# Remember to replace stub functions and paths with actual implementations and paths in your environment.
def load_parc_ids(parc_res_curr, medial_mask, parc_ids_path_format):
    current_parc = parc_ids_path_format.format(parc_res_curr)
    parc_ids     = nib.load(current_parc)
    
    parc_ids_array = np.array(parc_ids.dataobj).astype('float32')
    parc_ids_array = np.reshape(parc_ids_array.flatten(), (64984, 1))
    return parc_ids_array[medial_mask == 1].flatten()


In [ ]:
# Example usage
scan_ids = [1]
parc_res_currs = ['100']
gen_transforms = 0  # Set to 1 to generate transformations

perform_connectivity_hyperalignment(scan_ids, parc_res_currs, rest_indices, IDs_use_rest, gen_transforms, connectivity_paths)

In [ ]:
# Example usage
scan_ids = 1
parc_res_currs = ['100']
gen_transforms = 0  # Set to 1 to generate transformations
perform_piecewise_hyperalignment(scan_ids, parc_res_currs, movie_indices, IDs_use_movie, piecewise_paths,gen_transforms)